In [1]:
import sys
import os
import csv
from gpt_suite import gpt_mp_handler
import pandas as pd
import random
import json
from tqdm.auto import tqdm
import re
from datetime import datetime

In [2]:
criteria_dict = json.load(open('../../automated_verification/outputs/norm_violation/final_filtered_criteria.json'))

In [3]:
quantifier_msg = """In the first step of the process, you are a helpful quantifier assistant. You quantify the output of different tasks based on the given criteria.
The criterion is given in a dictionary format where each key is a distinct criteria.
The value of each key is a dictionary as follows {"description": criteria description , "accepted_values": possible accepted inputs for this key}
You are going to quantify each of the crieria for a given task based on the task decription.
Return a dictionary where the keys are the criteria and the values are the assessed performance based on accepted values for each criteria.
Return only the dictionary as a json format string and nothing else."""

In [4]:
DEFAULT_SYSTEM_PROMPT = """
### Task Overview:
As a two-step cultural and social norms analysis assistant, your task is to evaluate a provided conversation from the context of Chinese culture based on a given social norm and a corresponding norm concept. Your analysis should be comprehensive, considering factors such as the age, relationships, settings (e.g., work, family, friends), and the topic of the conversation.

### Steps for Analysis:

1. **Evaluate Social Norm and Norm Concept Compatibility**  
   - **Task:** Judge whether the provided social norm aligns with the given norm concept.
   - **Action:** State if the social norm **matches** or **doesn't match** the norm concept.
   - **Justification:** Provide a concise reason for your judgment.

2. **Assess Relevance to the Conversation**  
   - **Task:** Determine if the provided social norm and norm concept are relevant to the context of the conversation.
   - **Action:** State if the norm is **relevant** or **irrelevant**.
   - **Justification:** Provide a concise reason for your judgment.
  
3. **Determine Social Norm Violation**  
   - **Task:** Judge whether the social norm was **adhered to** or **violated** in the conversation.
   - **Justification:** Provide a concise reason for your judgment.
  
4. **Annotate Conversation-Specific Details**  
   - **Enactor Role:** Identify the social role of the person expected to follow the norm (e.g., parent, child, boss). Strictly do not provide names of the speakers here. Provide the specific social role of the enactor in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Acceptor Role:** Identify the social role of the person affected by the norm's adherence or violation (e.g., parent, child, employee). Strictly do not provide names of the speakers here. Provide the specific social role of the acceptor in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.

5. **Violation Analysis** *(Only if a violation occurred)*  
   If the norm was violated, provide the following additional details:
   - **Violating Action:** A brief description of the action that caused the violation (e.g., "badmouthing parents").
   - **Violator Role:** Social role of the person who violated the norm. Strictly do not provide names of the speakers here. Provide the specific social role of the violator in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Victim Role:** Social role of the person affected by the violation. Strictly do not provide names of the speakers here. Provide the specific social role of the victim in this situation here (For e.g: strictly do not say 'any family member', rather say spouse/wife/husband/child/family member). You may say unsure in highly uncertain cases.
   - **Violator Emotion:** Identify the violator's emotion using one of the following: **anger, fear, sadness, disgust, surprise, anticipation, trust, joy, neutral**.
   - **Victim Emotion:** Identify the victim's emotion using one of the following: **anger, fear, sadness, disgust, surprise, anticipation, trust, joy, neutral**.

### Response Format:
Your response must adhere to the format below:

Social Norm - Norm Concept Compatibility: <match/doesn't match>
Compatibility Justification: <short justification>

{Only if compatible}
Relevance: <relevant/irrelevant>
Relevance Justification: <short justification>

Enactor Role: <sitatuion specific social role, strictly not the name, of the person>
Acceptor Role: <sitatuion specific social role, strictly not the name, of the person>

Violation Status: <adhere/violate>
Violation Status Justification: <short justification>

{Only if a violation occurs}
Violating Action: <short phrase>
Violator Role: <sitatuion specific social role, strictly not the name, of the person>
Victim Role: <sitatuion specific social role, strictly not the name, of the person>
Violator Emotion: <one of 9 basic emotions>
Victim Emotion: <one of 9 basic emotions>
""".strip()

In [5]:
final_judgment = "Now given the above criteria and inputs provided above, provide judgments and annotation in the 'Response Format'. Strictly adhere to the 'Response Format' provided in the task instructions."

def symbolic_annotator(quantifier_prompts) -> list:
    config = {"temperature": 0, "max_tokens": 500}
    model = 'gpt-4o-mini'
    handler = gpt_mp_handler.GPTMPHandler(api_key=openai_key, gen_conf=config, num_worker=10)
    results = list()
    batch = []
    for norm_id in quantifier_prompts:
        ins = {
            'init_context': '',
            'questions': [quantifier_prompts[norm_id], final_judgment],
            'task_desc': DEFAULT_SYSTEM_PROMPT,
            'debug_log': debug_dir,
            'model_name': model
        }
        batch.append(ins)
        results.append([norm_id])
    handler.add_batch(batch)
    outs = handler.process()

    # print(outs)
    for idx, out in enumerate(outs):
        if len(out) == 0:
            print("ERROR:Missing...")
            results[idx].append(-1)
            continue
        for ques, resp in out.items():
            results[idx].append(resp)
    return results

In [6]:
symbolic_prompts = json.load(open('/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_prompts.json'))

In [7]:
symbolic_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_annotation_verification_outputs/'
bnames = os.listdir(symbolic_output_dir)
resps = {}
outs = {}
for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(symbolic_output_dir, bname)))
    for out in bres:
        n_id, _, ann, quality = out
        outs[n_id] = (ann.strip(), quality.strip()) 
            
print(len(outs))

  0%|          | 0/41 [00:00<?, ?it/s]

40816


In [8]:
print(len(symbolic_prompts[2]), len(outs))

40816 40816


In [16]:
quantifier_prompts = {}

prompts = symbolic_prompts[2]

for n_id in tqdm(prompts):
    test_case = prompts[n_id].strip()
    message = quantifier_msg + \
    "\n\nEvaluation dictionary: " + str(criteria_dict) + \
    "\n\nActual test case to evaluate:\n" + test_case
    quantifier_prompts[int(n_id)] = message.strip()

  0%|          | 0/40816 [00:00<?, ?it/s]

In [17]:
print(len(quantifier_prompts))

40816


In [19]:
openai_key = '<put-your-key-here>'

In [20]:
debug_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_agenteval_logs/'
os.makedirs(debug_dir, exist_ok=True)

In [24]:
quantifier_prompt_keys = sorted(list(quantifier_prompts.keys()))

batch_id = 0
batches = []
b = 0
bsz = 1000
e = bsz
while b < len(quantifier_prompt_keys):
    batch_keys = quantifier_prompt_keys[b:e]
    batch = {}
    for bkey in batch_keys:
        batch[bkey] = quantifier_prompts[bkey]
    batches.append(batch)
    b += bsz
    e += bsz

print(len(batches))

41


In [25]:
output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_agenteval_outputs/'
os.makedirs(output_dir, exist_ok=True)

t1 = datetime.now()
for i, batch in tqdm(enumerate(batches)):
    batch_out_path = os.path.join(output_dir, f"batch_{i}.json")
    if not os.path.exists(batch_out_path):
        batch_res = symbolic_annotator(batch)
        json.dump(batch_res, open(batch_out_path, 'w'))
    t2 = datetime.now()
    print(f'batch {i} done.', t2-t1)

0it [00:00, ?it/s]

Verifying Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 0 done. 0:07:27.508561


Verifying Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

...

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 39 done. 7:51:51.699833


Verifying Batch:   0%|          | 0/816 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/816 [00:00<?, ?it/s]

batch 40 done. 8:01:47.582550


In [26]:
json.dump([criteria_dict, quantifier_msg, DEFAULT_SYSTEM_PROMPT, final_judgment, quantifier_prompts], open('/homes/rpujari/scratch_ml/DARPA/automated_verification/symbolic_agenteval_prompts.json', 'w'))